In [2]:
from ovo import schedulers
import os
import time

Registering plugin ovo_promb
Registering plugin ovo_proteindj


OVO home /home/username/ovo

In [3]:
# Download input file
# !wget -O ../../data/inputs/1A4I.pdb https://files.rcsb.org/download/1A4I.pdb

## Choose a scheduler

In [4]:
for scheduler_key, scheduler in schedulers.items():
    print(scheduler_key, scheduler.submission_args);

pbs_singularity {'profile': 'singularity', 'config': '/home/username/ovo/nextflow_pbs_singularity.config'}
local_singularity {'profile': 'singularity,cpu_env', 'max_memory': '8GB', 'config': '/home/username/ovo/nextflow_local.config'}
local_conda {'profile': 'conda,cpu_env', 'max_memory': '8GB', 'config': '/home/username/ovo/nextflow_local.config'}
local_single_gpu {'profile': 'singularity', 'max_memory': '15GB', 'config': '/home/username/ovo/nextflow_local.config'}


In [5]:
scheduler = schedulers['local_single_gpu']
print(scheduler.submission_args)

{'profile': 'singularity', 'max_memory': '15GB', 'config': '/home/username/ovo/nextflow_local.config'}


## Submit workflow

In [6]:
job_id = scheduler.submit(
    "rfdiffusion-end-to-end",
    params={
        "batch_size": 200,
        "rfdiffusion_input_pdb": "../../data/inputs/1A4I.pdb", 
        "rfdiffusion_num_designs": 1, 
        "rfdiffusion_contig": "10-35/A56-56/10-35/A100-100/10-35/A125-125/10-35", 
        "design_type": "scaffold", 
        "mpnn_num_sequences": 2, 
        #"mpnn_fastrelax_cycles": 3, 
        "refolding_tests": "af2_model_1_ptm_ft_3rec", 
        "rfdiffusion_run_parameters": "diffuser.T=50", 
        "mpnn_run_parameters": "--omit_AA CX --temperature 0.1", 
    },
    submission_args=dict(
        # optional args for scheduler 
    )
)

Submitting workflow: nextflow run -with-trace trace.txt -work-dir /home/username/ovo/workdir/work /home/username/projects/ovo-open-source/ovo/ovo/pipelines/rfdiffusion-end-to-end --publish_dir output --reference_files_dir /home/username/ovo/reference_files --shared_modules ovo:/home/username/projects/ovo-open-source/ovo/ovo,ovo_promb:/home/username/projects/ovo-open-source/ovo-promb/ovo_promb,ovo_proteindj:/home/username/projects/ovo-proteindj/ovo_proteindj -config /home/username/projects/ovo-open-source/ovo/ovo/pipelines/nextflow_default.config -config /home/username/projects/ovo-open-source/ovo/ovo/pipelines/rfdiffusion-end-to-end/nextflow.config -profile singularity -config /home/username/ovo/nextflow_local.config --max_memory 15GB -ansi-log false -bg --batch_size 200 --rfdiffusion_input_pdb /home/username/projects/ovo-open-source/ovo-examples/jupyter_notebooks_example/data/inputs/1A4I.pdb --rfdiffusion_num_designs 1 --rfdiffusion_contig 10-35/A56-56/10-35/A100-100/10-35/A125-125/10

In [13]:
print(job_id)

9bb9ef98-c06a-11f0-8215-029f0fc8e1cf


In [8]:
print(scheduler.get_status_label(job_id))

Running


In [9]:
print(scheduler.get_result(job_id))

None


In [10]:
result = scheduler.wait(job_id)

if not result:
    print(scheduler.get_log(job_id))
    assert False, 'Job failed'

In [14]:
output_dir = scheduler.get_output_dir(job_id)

!ls {output_dir}/contig1_batch1/

af2_model_1_ptm_ft_3rec        rfdiffusion_pdb
af2_model_1_ptm_ft_3rec.jsonl  rfdiffusion_standardized_pdb
backbone_metrics.csv	       rfdiffusion_traj
backbones_filtered	       rfdiffusion_trb
ligandmpnn		       seq_composition.csv


In [15]:
!head -15 {output_dir}/contig1_batch1/rfdiffusion_standardized_pdb/contig1_batch1_0_standardized.pdb

REMARK   1 Input contig: "10-35/A56-56/10-35/A100-100/10-35/A125-1"             
REMARK   1 Input contig: "25/10-35"                                             
REMARK   1 Standardized contig: "35-35/A56-56/28-28/A100-100/32-32/A125-1"      
REMARK   1 Standardized contig: "25/11-11"                                      
REMARK   1 Chains: "A"                                                          
REMARK   1 Input hotspots:                                                      
REMARK   1 Standardized hotspots:                                               
ATOM      1  N   GLY A   1      22.670   6.728 -17.258  1.00  0.00              
ATOM      2  CA  GLY A   1      21.700   7.644 -16.671  1.00  0.00              
ATOM      3  C   GLY A   1      20.490   6.893 -16.130  1.00  0.00              
ATOM      4  O   GLY A   1      19.958   7.233 -15.073  1.00  0.00              
ATOM      5  N   GLY A   2      20.099   5.956 -16.820  1.00  0.00              
ATOM      6  CA  GLY A   2  